In [ ]:
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def find_project_root():
    candidates = []
    if "__file__" in globals():
        candidates.append(Path(__file__).resolve().parents[3])

    cwd = Path.cwd().resolve()
    candidates.extend([
        cwd,
        cwd / "Grad_Research_new",
        cwd / "Grad_Research",
        cwd.parent,
        Path("/content/Grad_Research_new"),
        Path("/content/Grad_Research"),
        Path("/content/drive/MyDrive/Grad_Research_new"),
        Path("/content/drive/MyDrive/Grad_Research"),
        Path("/content/drive/MyDrive/Colab Notebooks/Grad_Research"),
    ])

    for root in candidates:
        if (root / "data").is_dir():
            return root

    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception:
        pass

    for root in candidates:
        if (root / "data").is_dir():
            return root

    raise FileNotFoundError(
        "Could not find the project root. "
        "In Colab, place the repository at MyDrive/Grad_Research_new or /content/Grad_Research_new."
    )


ROOT       = find_project_root()
EXP001_DIR = ROOT / "EXP001"
MODEL_DIR  = EXP001_DIR / "models"

print(f"Device      : {device}")
print(f"Project root: {ROOT}")
print(f"Model dir   : {MODEL_DIR}")

In [ ]:
@dataclass
class Config:
    # DRQN
    embed_dim:    int   = 16
    lstm_hidden:  int   = 64
    dropout_rate: float = 0.0

    # Training
    test_length:         int   = 40
    gamma:               float = 0.1
    memory_capacity:     int   = 200
    epsilon:             float = 0.1
    batch_size:          int   = 32
    q_network_iteration: int   = 40
    learning_rate:       float = 1e-3
    validation_interval: int   = 200

    # Dataset
    dataset_name: str = "ShinyItemAnalysis_dataMedical"

    # Evaluation
    network_id: str = "subject_200"

In [ ]:
from typing import Any, cast
from scipy.optimize import minimize_scalar


def FI(item_para, theta, D=1):
    a = item_para[:, 0]
    b = item_para[:, 1]
    c = item_para[:, 2]
    info = (
        D**2 * a**2 * (1 - c)
        / (c + np.exp(D * a * (theta - b)))
        / (1 + np.exp(-D * a * (theta - b))) ** 2
    )
    return info


def MLE(item_paras, resp, D=1):
    a = item_paras[:, 0]
    b = item_paras[:, 1]
    c = item_paras[:, 2]

    def mins_likelihood(x):
        logl = 0
        for i in range(len(resp)):
            p = (1 - c[i]) / (1 + np.exp(-D * a[i] * (x - b[i]))) + c[i]
            p = np.clip(p, 1e-10, 1 - 1e-10)
            logl -= resp[i] * np.log(p) + (1 - resp[i]) * np.log(1 - p)
        return logl

    result = cast(Any, minimize_scalar(mins_likelihood, bounds=(-4, 4), method="bounded"))
    return np.array(result.x).reshape(1,)


def MLE_TEST(item_paras, resp, D=1):
    def mins_likelihood(x):
        logl = 0
        for i in range(resp_i.shape[0]):
            p = (1 - c[i]) / (1 + np.exp(-D * a[i] * (x - b[i]))) + c[i]
            p = np.clip(p, 1e-10, 1 - 1e-10)
            logl -= resp_i[i] * np.log(p) + (1 - resp_i[i]) * np.log(1 - p)
        return logl

    theta = np.zeros(resp.shape[1])
    for i in range(resp.shape[1]):
        resp_i = resp[:, i]
        a = item_paras[:, i, 0]
        b = item_paras[:, i, 1]
        c = item_paras[:, i, 2]
        result = cast(Any, minimize_scalar(mins_likelihood, bounds=(-4, 4), method="bounded"))
        theta[i] = result.x
    return np.expand_dims(theta, axis=0)


def Apply_Positive_Constraint(model, min_value=0.0):
    for param in model.parameters():
        param.data = torch.clamp(param.data, min=min_value)

In [ ]:
class DRQN(nn.Module):
    def __init__(self, action_space, embed_dim, lstm_hidden, dropout_rate):
        super(DRQN, self).__init__()
        self.embed        = nn.Embedding(3, embed_dim)  # 0=wrong, 1=correct, 2=start
        self.lstm         = nn.LSTM(embed_dim, lstm_hidden, batch_first=True)
        self.out          = nn.Linear(lstm_hidden, action_space)
        self.dropout      = nn.Dropout(dropout_rate)
        self.action_space = action_space
        self.embed_dim    = embed_dim
        self.lstm_hidden  = lstm_hidden

    def forward(self, resps, hidden=None):
        # resps: (batch, seq_len) long, values in {0, 1, 2}
        x = self.embed(resps)
        x = self.dropout(x)
        out, hidden = self.lstm(x, hidden)
        out = self.dropout(out)
        return self.out(out), hidden

    def init_hidden(self, batch_size=1):
        h = torch.zeros(1, batch_size, self.lstm_hidden, device=device)
        c = torch.zeros(1, batch_size, self.lstm_hidden, device=device)
        return (h, c)

    def initialize(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
            elif isinstance(m, nn.LSTM):
                for name, param in m.named_parameters():
                    if "weight" in name:
                        nn.init.kaiming_normal_(param)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, mean=0.0, std=0.1)

In [ ]:
def Choose_Action(prev_resp_t, hidden, item_id_arr, epsilon):
    with torch.no_grad():
        q_value, hidden = eval_net(prev_resp_t, hidden)
        # FIX: use rand() (uniform) instead of randn() (normal) for correct epsilon-greedy rate
        if np.random.rand() >= epsilon:
            qv = q_value.squeeze(0).squeeze(0).clone()
            if item_id_arr.size > 0:
                qv[torch.from_numpy(item_id_arr).to(device).long()] = -float("inf")
            action = int(qv.argmax().cpu().numpy())
        else:
            # FIX: use item_id_arr.size > 0 instead of any(item_id_arr) to correctly
            # detect non-empty arrays even when item 0 is the only selected item
            if item_id_arr.size > 0:
                action = int(
                    np.random.choice(np.delete(np.arange(action_space), item_id_arr))
                )
            else:
                action = int(np.random.choice(np.arange(action_space)))
    return action, hidden


def Choose_Action_Test(prev_resps_t, hidden, item_id_history):
    with torch.no_grad():
        q_value, hidden = eval_net(prev_resps_t, hidden)
        q_value = q_value.squeeze(1).cpu().numpy()
        if item_id_history.shape[0] > 0:
            q_value[
                np.tile(np.arange(item_id_history.shape[1])[np.newaxis, :], (item_id_history.shape[0], 1)),
                item_id_history,
            ] = -np.inf
        action = q_value.argmax(axis=1)
    return action, hidden

In [ ]:
def TRAIN(cfg, training_size=None, validation_size=None):

    loss_func = nn.MSELoss()
    eval_net.train()
    optimizer = optim.Adam(eval_net.parameters(), lr=cfg.learning_rate)

    memory: list = []
    memory_idx = 0
    learn_step_counter = 0

    if training_size is None:
        training_size = train_valid_resp.shape[0]
    if validation_size is None:
        validation_size = min(200, training_size)

    for j in range(training_size):
        prev_resp_t = torch.tensor([[START_TOKEN]], dtype=torch.long, device=device)
        hidden = eval_net.init_hidden(1)

        item_id_arr = np.array([]).astype("int64")
        resp_arr    = np.array([]).astype("int64")
        theta_current = np.random.rand(1) - 0.5

        ep_resps   = [START_TOKEN]
        ep_actions = []
        ep_rewards = []

        for i in range(cfg.test_length):
            action, hidden = Choose_Action(prev_resp_t, hidden, item_id_arr, cfg.epsilon)

            response = int(train_valid_resp[j, action])
            reward   = FI(item_bank[np.array([action]),], theta_current[-1])

            item_id_arr = np.concatenate((item_id_arr, np.array([action])))
            resp_arr    = np.concatenate((resp_arr,    np.array([response])))

            if len(np.unique(resp_arr)) == 1:
                if response == 1:
                    theta_current = np.array(
                        [theta_current[-1] + (item_bank[:, 1].max() - theta_current[-1]) / 2]
                    )
                else:
                    theta_current = np.array(
                        [theta_current[-1] - (theta_current[-1] - item_bank[:, 1].min()) / 2]
                    )
            else:
                theta_current = MLE(item_bank[item_id_arr,], resp_arr)

            ep_resps.append(response)
            ep_actions.append(action)
            ep_rewards.append(float(reward[0]))

            prev_resp_t = torch.tensor([[response]], dtype=torch.long, device=device)

        episode = {
            "resps":   np.asarray(ep_resps,   dtype=np.int64),    # length T+1
            "actions": np.asarray(ep_actions, dtype=np.int64),    # length T
            "rewards": np.asarray(ep_rewards, dtype=np.float32),  # length T
        }
        if len(memory) < cfg.memory_capacity:
            memory.append(episode)
        else:
            memory[memory_idx] = episode
        memory_idx = (memory_idx + 1) % cfg.memory_capacity

        if len(memory) >= cfg.batch_size:
            indices = np.random.choice(len(memory), cfg.batch_size, replace=False)
            batch   = [memory[i] for i in indices]

            resps_t   = torch.LongTensor(np.stack([ep["resps"]   for ep in batch])).to(device)
            actions_t = torch.LongTensor(np.stack([ep["actions"] for ep in batch])).to(device)
            rewards_t = torch.FloatTensor(np.stack([ep["rewards"] for ep in batch])).to(device)

            q_full_eval, _ = eval_net(resps_t)
            with torch.no_grad():
                q_full_target, _ = target_net(resps_t)

            q_eval = q_full_eval[:, :-1, :].gather(2, actions_t.unsqueeze(-1)).squeeze(-1)

            q_next_all = q_full_target[:, 1:, :].clone()
            selected_mask = torch.cumsum(
                F.one_hot(actions_t, num_classes=action_space), dim=1
            ).bool()
            q_next_all[selected_mask] = -float("inf")
            q_next = q_next_all.max(dim=2)[0]

            is_terminal = torch.zeros_like(rewards_t)
            is_terminal[:, -1] = 1.0
            q_target = rewards_t + cfg.gamma * q_next * (1.0 - is_terminal)

            loss = loss_func(q_eval, q_target)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            # FIX: apply positive constraint after optimizer.step() so the clamp
            # acts on the updated weights, not the pre-update weights
            Apply_Positive_Constraint(eval_net)

            learn_step_counter += 1
            if learn_step_counter % cfg.q_network_iteration == 0:
                target_net.load_state_dict(eval_net.state_dict())

        ### Validation ###
        if (j + 1) % cfg.validation_interval == 0:
            eval_net.eval()
            valid_bias = np.zeros((cfg.test_length, validation_size))

            validation_id = np.random.choice(
                training_size, validation_size, replace=False
            )
            valid_theta = train_valid_theta[validation_id]
            valid_resp  = train_valid_resp[validation_id, :]

            theta_state = np.random.rand(validation_size) - 0.5

            prev_resps_t = torch.full(
                (validation_size, 1), START_TOKEN, dtype=torch.long, device=device
            )
            valid_hidden = eval_net.init_hidden(validation_size)

            item_id_history = np.empty((0, validation_size), dtype=np.int64)
            resp_history    = np.empty((0, validation_size), dtype=np.int64)

            for i in range(cfg.test_length):
                action, valid_hidden = Choose_Action_Test(
                    prev_resps_t, valid_hidden, item_id_history
                )
                response = valid_resp[np.arange(validation_size), action].astype(np.int64)

                item_id_history = np.concatenate((item_id_history, action[np.newaxis, :]))
                resp_history    = np.concatenate((resp_history,    response[np.newaxis, :]))

                theta_0  = np.zeros(validation_size)
                idx_full = np.sum(resp_history, axis=0) == resp_history.shape[0]
                idx_zero = np.sum(resp_history, axis=0) == 0
                idx_norm = np.bitwise_not(idx_full | idx_zero)
                theta_0[idx_full] = (
                    theta_state[idx_full]
                    + (item_bank[:, 1].max() - theta_state[idx_full]) / 2
                )
                theta_0[idx_zero] = (
                    theta_state[idx_zero]
                    + (item_bank[:, 1].min() - theta_state[idx_zero]) / 2
                )
                theta_0[idx_norm] = np.squeeze(
                    MLE_TEST(item_bank[item_id_history[:, idx_norm]], resp_history[:, idx_norm])
                )

                theta_state = theta_0
                valid_bias[i] = theta_0 - valid_theta

                prev_resps_t = torch.from_numpy(response).long().unsqueeze(1).to(device)

            step_valid = np.transpose(
                np.vstack((
                    np.arange(1, cfg.test_length + 1),
                    np.mean(valid_bias, axis=1),
                    np.sqrt(np.mean(valid_bias ** 2, axis=1)),
                    np.mean(abs(valid_bias), axis=1),
                ))
            )
            print("subject: {}\n\n{}\n".format(j + 1, step_valid))

            MODEL_DIR.mkdir(parents=True, exist_ok=True)
            torch.save(
                eval_net,
                MODEL_DIR / (
                    "drqn_" + cfg.dataset_name + "_gamma_" + str(cfg.gamma)
                    + "_subject_" + str(j + 1) + ".t7"
                ),
            )
            TEST(
                cfg,
                valid_theta,
                validation_size,
                valid_resp,
                output_id="subject_" + str(j + 1),
            )

            eval_net.train()

In [ ]:
def TEST(cfg, theta_test, testing_size=None, response_data=None, output_id=1):

    with torch.no_grad():
        eval_net.eval()

        if response_data is None:
            response_data = test_resp
        if testing_size is None:
            testing_size = response_data.shape[0]

        theta_state = np.random.rand(testing_size) - 0.5

        prev_resps_t = torch.full(
            (testing_size, 1), START_TOKEN, dtype=torch.long, device=device
        )
        test_hidden = eval_net.init_hidden(testing_size)

        item_id_history = np.empty((0, testing_size), dtype=np.int64)
        resp_history    = np.empty((0, testing_size), dtype=np.int64)
        theta           = np.empty((0, testing_size))
        dqn_step        = np.zeros((1, 4))

        for i in range(cfg.test_length):
            action, test_hidden = Choose_Action_Test(
                prev_resps_t, test_hidden, item_id_history
            )
            response = response_data[np.arange(testing_size), action].astype(np.int64)

            item_id_history = np.concatenate((item_id_history, action[np.newaxis, :]))
            resp_history    = np.concatenate((resp_history,    response[np.newaxis, :]))

            theta_0  = np.zeros([1, testing_size])
            idx_full = np.sum(resp_history, axis=0) == resp_history.shape[0]
            idx_zero = np.sum(resp_history, axis=0) == 0
            idx_norm = np.bitwise_not(idx_full | idx_zero)
            theta_0[:, idx_full] = (
                theta_state[idx_full] + (item_bank[:, 1].max() - theta_state[idx_full]) / 2
            )
            theta_0[:, idx_zero] = (
                theta_state[idx_zero] + (item_bank[:, 1].min() - theta_state[idx_zero]) / 2
            )
            theta_0[:, idx_norm] = MLE_TEST(
                item_bank[item_id_history[:, idx_norm]], resp_history[:, idx_norm]
            )

            theta = np.concatenate((theta, theta_0))
            theta_state = theta_0[0]

            dqn_step = np.vstack([
                dqn_step,
                np.array([
                    i + 1,
                    np.mean(theta_0 - theta_test),
                    np.sqrt(np.mean((theta_0 - theta_test) ** 2)),
                    np.mean(abs(theta_0 - theta_test)),
                ]),
            ])
            print("step {:g}, bias {:.3f}, rmse {:.3f}, mae {:.3f}".format(
                dqn_step[-1, 0], dqn_step[-1, 1], dqn_step[-1, 2], dqn_step[-1, 3]
            ))

            prev_resps_t = torch.from_numpy(response).long().unsqueeze(1).to(device)

        user_id        = np.repeat(np.arange(1, testing_size + 1), cfg.test_length).reshape(-1, 1)
        step           = np.tile(np.arange(1, cfg.test_length + 1), testing_size).reshape(-1, 1)
        item_id_out    = (item_id_history + 1).transpose().reshape(-1, 1)
        resp_out       = resp_history.transpose().reshape(-1, 1)
        theta_true_col = np.repeat(theta_test, cfg.test_length).reshape(-1, 1)
        theta_est      = theta.transpose().reshape(-1, 1)
        bias           = (theta - theta_test).transpose().reshape(-1, 1)
        dqn_data = np.hstack(
            [user_id, step, item_id_out, resp_out, theta_true_col, theta_est, bias]
        )
        dqn_data = pd.DataFrame(dqn_data).rename(
            columns={
                0: "userID",
                1: "step",
                2: "itemID",
                3: "resp",
                4: "theta_true",
                5: "theta_est",
                6: "bias",
            }
        )
        stem = f"real_bank_responses_DRQN_gamma_{cfg.gamma}_{output_id}"
        RESULTS_DIR.mkdir(parents=True, exist_ok=True)
        dqn_data.to_csv(RESULTS_DIR / f"records_{stem}.csv", index=False)

        summary_rows = []
        for s, grp in dqn_data.groupby("step"):
            b = grp["bias"]
            r = (
                grp["theta_true"].corr(grp["theta_est"])
                if grp["theta_true"].std() > 0 and grp["theta_est"].std() > 0
                else float("nan")
            )
            summary_rows.append({
                "step": s,
                "Bias": b.mean(),
                "RMSE": np.sqrt((b ** 2).mean()),
                "MAE":  b.abs().mean(),
                "r":    r,
            })
        pd.DataFrame(summary_rows).to_csv(RESULTS_DIR / f"summary_{stem}.csv", index=False)
        print(f"\nSaved to {RESULTS_DIR}")

In [ ]:
cfg = Config(
    embed_dim           = 16,
    lstm_hidden         = 64,
    dropout_rate        = 0.0,
    test_length         = 40,
    gamma               = 0.1,
    memory_capacity     = 200,
    epsilon             = 0.1,
    batch_size          = 32,
    q_network_iteration = 40,
    learning_rate       = 1e-3,
    validation_interval = 200,
    dataset_name        = "ShinyItemAnalysis_dataMedical",
    network_id          = "subject_200",
)

DATA_DIRS = {
    "ShinyItemAnalysis_dataMedical": EXP001_DIR / "data" / "real_responses_ShinyItemAnalysis_dataMedical",
    "TAM_data_ctest2":               EXP001_DIR / "data" / "real_responses_TAM_data_ctest2",
}
DATA_DIR    = DATA_DIRS[cfg.dataset_name]
RESULTS_DIR = EXP001_DIR / "results" / cfg.dataset_name
START_TOKEN = 2

item_bank    = np.array(pd.read_csv(DATA_DIR / "real item bank.csv")[["a", "b", "c"]])
action_space = item_bank.shape[0]

train_valid_resp  = np.array(pd.read_csv(DATA_DIR / "real responses for training.csv"))
train_valid_theta = np.array(pd.read_csv(DATA_DIR / "true theta for training.csv")).reshape(-1)

print(f"Dataset     : {cfg.dataset_name}")
print(f"Data dir    : {DATA_DIR}")
print(f"Results dir : {RESULTS_DIR}")
print(f"item bank   : {item_bank.shape}")
print(f"train resp  : {train_valid_resp.shape}")
print(f"train theta : {train_valid_theta.shape}")
print(f"\nConfig:\n{cfg}")

In [ ]:
eval_net   = DRQN(action_space, cfg.embed_dim, cfg.lstm_hidden, cfg.dropout_rate).to(device)
target_net = DRQN(action_space, cfg.embed_dim, cfg.lstm_hidden, cfg.dropout_rate).to(device)

eval_net.initialize()
target_net.initialize()
target_net.load_state_dict(eval_net.state_dict())

TRAIN(cfg)

In [ ]:
test_resp  = np.array(pd.read_csv(DATA_DIR / "real responses for testing.csv"))
theta_test = np.array(pd.read_csv(DATA_DIR / "true theta for testing.csv")).reshape(-1)

eval_net = DRQN(action_space, cfg.embed_dim, cfg.lstm_hidden, cfg.dropout_rate).to(device)
eval_net = torch.load(
    MODEL_DIR / f"drqn_{cfg.dataset_name}_gamma_{cfg.gamma}_{cfg.network_id}.t7",
    weights_only=False,
)

TEST(cfg, theta_test)